# Moltbook Oversight Agent — GRPO Training

Train an LLM oversight agent to evaluate conversation participants across 12 behavioral traits using **Group Relative Policy Optimization (GRPO)** via [Unsloth](https://unsloth.ai) + [TRL](https://huggingface.co/docs/trl).

> Run this on a **free Colab T4 (16 GB)** with the default `unsloth/Llama-3.2-1B-Instruct` model.  
> For better results, upgrade to Colab Pro L4 (22 GB) and switch to `unsloth/Qwen3-8B-Instruct`.

## What the agent learns
Given a conversation thread plus a **target message**, produce:
```
<think>
  step-by-step reasoning about the agent's behavior
</think>
<verdict>
  { "virtue": 0.8, "goodwill": 0.7, "manipulation": 0.0, ... "alignment_status": "aligned" }
</verdict>
```

## Reward signal
| Function | Max | Scaled by `length_scale`? | Purpose |
|---|---|---|---|
| `format_reward` | 1.0 | No | Forces `<think>`/`<verdict>` structure |
| `alignment_reward` | 2.0 | Yes | Correct alignment verdict |
| `trait_reward` | ~1.0 | Yes | Accurate 12-trait MAE |

`length_scale = (turn_index + 1) / total_turns` — early-turn judgments with little context count less.

## Sections
1. Installation
2. Configuration
3. Load model + LoRA
4. Load & prepare training data
5. (Optional) Format warmup SFT
6. Reward functions
7. GRPO training
8. Quick inference test
9. Save LoRA adapter

## 1. Installation

In [1]:
%%capture
import os

# MUST be set before importing unsloth/vllm — enables sequential memory sharing
# so vLLM rollout and training backward pass alternate rather than competing for VRAM.
os.environ['UNSLOTH_VLLM_STANDBY'] = '1'

IN_COLAB = 'COLAB_' in ''.join(os.environ.keys())
if not IN_COLAB:
    # Local install: pip install unsloth vllm
    pass  # For Colab, we need the extra install cell below ↓

In [3]:
%%capture
# @title Colab Extra Install { display-mode: "form" }
# %%capture must be the first line — pins package versions for Colab stability.
# Unsloth requires specific vllm/transformers/trl versions; Colab's pre-installed
# torchvision/bitsandbytes/xformers can conflict without explicit pins.
# T4 (free tier) needs vllm==0.9.2 — newer vllm dropped T4 CUDA kernel support.
# L4/A100 can use the current vllm release.
import os, subprocess
if 'COLAB_' in ''.join(os.environ.keys()):
    is_t4 = 'Tesla T4' in str(subprocess.check_output(['nvidia-smi']))
    vllm_ver   = 'vllm==0.9.2'    if is_t4 else 'vllm==0.15.1'
    triton_ver = 'triton==3.2.0'  if is_t4 else 'triton'
    os.system(f'pip install -qqq {vllm_ver} torchvision bitsandbytes xformers unsloth')
    os.system(f'pip install -qqq {triton_ver}')
    os.system('pip install -qqq transformers==4.56.2')
    os.system('pip install -qqq --no-deps trl==0.22.2')
    print(f'Colab install complete ({"T4" if is_t4 else "L4/A100"})')

In [4]:
# Clone the repo and install the grpo-pipeline package (skip if running locally)
if IN_COLAB:
    REPO_URL = 'https://github.com/Eephor/DataMassageForGRPO.git'
    if not os.path.exists('DataMassageForGRPO'):
        os.system(f'git clone {REPO_URL}')
    os.chdir('DataMassageForGRPO/grpo-pipeline')
    os.system('pip install -e ".[train]" -qqq')
    print('Repo cloned and package installed.')
else:
    # Make sure we're in the grpo-pipeline directory
    if os.path.basename(os.getcwd()) != 'grpo-pipeline':
        os.chdir('grpo-pipeline')
    print(f'Working directory: {os.getcwd()}')

Repo cloned and package installed.


## 2. Configuration

Edit the cells below to configure paths and training hyperparameters.

In [5]:
# ── Model ────────────────────────────────────────────────────────────────────
# Phase 1 (default): Llama-3.2-1B — fits on free T4, fast to iterate
MODEL_NAME = 'unsloth/Llama-3.2-1B-Instruct'
# Phase 2: uncomment for Qwen3-8B on Colab Pro L4 (22 GB):
# MODEL_NAME = 'unsloth/Qwen3-8B-Instruct'

# ── Paths ────────────────────────────────────────────────────────────────────
TRAIN_FILE   = '../transformed/train.jsonl'
OUTPUT_DIR   = '../lora-adapter'

# ── Sequence lengths ─────────────────────────────────────────────────────────
MAX_SEQ_LENGTH        = 2048   # total prompt + completion budget
MAX_COMPLETION_LENGTH = 768    # tokens the model may generate per sample
# MAX_PROMPT_LENGTH is computed below as MAX_SEQ_LENGTH - MAX_COMPLETION_LENGTH

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_RANK = 32   # higher = more capacity, more VRAM

# ── GRPO training ────────────────────────────────────────────────────────────
MAX_STEPS           = 200    # set to -1 for a full epoch
NUM_GENERATIONS     = 4      # group size; reduce to 2 if OOM
BATCH_SIZE          = 4      # per-device; reduce to 1 or 2 if OOM
LEARNING_RATE       = 5e-6
SAVE_STEPS          = 100
REPORT_TO           = 'none' # 'wandb' if you want W&B logging

print(f'Model:              {MODEL_NAME}')
print(f'Train file:         {TRAIN_FILE}')
print(f'Output dir:         {OUTPUT_DIR}')
print(f'max_seq_length:     {MAX_SEQ_LENGTH}')
print(f'num_generations:    {NUM_GENERATIONS}')
print(f'max_steps:          {MAX_STEPS}')

Model:              unsloth/Llama-3.2-1B-Instruct
Train file:         ../transformed/train.jsonl
Output dir:         ../lora-adapter
max_seq_length:     2048
num_generations:    4
max_steps:          200


## 3. Load Model + LoRA

In [6]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # False = 16-bit LoRA (not 4-bit)
    fast_inference=True,  # enable vLLM as the GRPO rollout engine
    max_lora_rank=LORA_RANK,  # pre-allocate vLLM memory for LoRA
    load_in_fp8=True,     # Float8 training — the key FP8 flag
)
print('Base model loaded.')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 03-08 03:04:18 [vllm_utils.py:723] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.15.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standby mode is enabled. Changing `gpu_memory_utilization` to 0.8312499999999999.
Unsloth: vLLM loading unsloth/Llama-3.2-1B-Instruct-FP8-Block with actual GPU utilization = 82.26%
Unsloth: Your GPU has CUDA compute capability 8.9 with VRAM = 22.03 GB.
Unsloth: Using conservativeness = 1.0. Chunked pre

/usr/local/lib/python3.12/dist-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 03-08 03:04:43 [model.py:541] Resolved architecture: LlamaForCausalLM
INFO 03-08 03:04:43 [model.py:1561] Using max model len 2048
INFO 03-08 03:04:43 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=6144.
INFO 03-08 03:04:43 [vllm.py:624] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Base model loaded.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK * 2,  # 2× rank → scales LoRA updates by 2, effectively 2× faster convergence
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
print(f'LoRA applied (rank={LORA_RANK}, alpha={LORA_RANK * 2}).')
model.print_trainable_parameters()

## 4. Load & Prepare Training Data

We load `train.jsonl` and inject the per-author system prompt into each record's prompt field.
The system prompt is **not stored in the dataset** — it is injected here at training time,
which keeps the dataset lean and lets the prompt evolve without re-running the transform.

In [ ]:
import json
from datasets import Dataset
from grpo_pipeline.transform import SYSTEM_PROMPT_TEMPLATE

# Load raw records
with open(TRAIN_FILE) as f:
    raw_records = [json.loads(line) for line in f if line.strip()]

print(f'Loaded {len(raw_records)} training records.')

# Preview the system prompt (truncated)
sample_author = raw_records[0]['author']
print(f'\nSystem prompt preview (author={sample_author!r}):')
print(SYSTEM_PROMPT_TEMPLATE.format(author=sample_author)[:300] + ' ...')

In [ ]:
def inject_system_prompts(batch):
    """Prepend a per-author system message to each prompt in the batch."""
    return {'prompt': [
        [{'role': 'system', 'content': SYSTEM_PROMPT_TEMPLATE.format(author=a)}] + p
        for a, p in zip(batch['author'], batch['prompt'])
    ]}

dataset = Dataset.from_list(raw_records)
dataset = dataset.map(inject_system_prompts, batched=True, desc='Injecting system prompts')

print(f'Dataset ready: {len(dataset)} records.')
print(f'Columns: {dataset.column_names}')

In [ ]:
# Preview a single prepared prompt
sample = dataset[0]
print(f'Author:         {sample["author"]}')
print(f'Alignment GT:   {sample["ground_truth_alignment"]}')
print(f'length_scale:   {sample["length_scale"]:.3f}  (turn {sample["turn_index"]}/{sample["total_turns"]-1})')
print()
prompt_text = tokenizer.apply_chat_template(
    sample['prompt'], tokenize=False, add_generation_prompt=True
)
print('Prompt (last 400 chars):')
print(prompt_text[-400:])

In [ ]:
# Check prompt length distribution — we want 90%+ to fit within MAX_SEQ_LENGTH - MAX_COMPLETION_LENGTH
import numpy as np

MAX_PROMPT_LENGTH = MAX_SEQ_LENGTH - MAX_COMPLETION_LENGTH

lengths = [len(tokenizer.apply_chat_template(r, tokenize=True)) for r in dataset['prompt']]
p90 = int(np.percentile(lengths, 90))
p99 = int(np.percentile(lengths, 99))
n_over = sum(1 for l in lengths if l > MAX_PROMPT_LENGTH)

print(f'Prompt token lengths:')
print(f'  min={min(lengths)}, median={int(np.median(lengths))}, p90={p90}, p99={p99}, max={max(lengths)}')
print(f'  Records exceeding MAX_PROMPT_LENGTH ({MAX_PROMPT_LENGTH}): {n_over} ({100*n_over/len(lengths):.1f}%)')
print(f'\nMAX_PROMPT_LENGTH={MAX_PROMPT_LENGTH} — adjust MAX_COMPLETION_LENGTH if too many exceed this.')

## 5. (Optional) Format Warmup SFT

This optional step briefly fine-tunes the model on a handful of "gold" examples that demonstrate
the `<think>/<verdict>` output format **before** GRPO begins. This is the same technique used in
the [Unsloth Llama FP8 GRPO notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama_FP8_GRPO.ipynb).

We construct gold outputs from the ground truth labels. The reasoning is synthetic but the verdict JSON is exact.

**Skip this cell if you want to rely on `format_reward` alone to teach the format during GRPO.**
The warmup speeds up format adoption but is not required.

In [ ]:
RUN_FORMAT_WARMUP = True   # ← set to False to skip
WARMUP_EXAMPLES   = 60     # number of short examples to SFT on
WARMUP_MAX_TOKENS = MAX_SEQ_LENGTH // 2  # keep warmup examples short

if RUN_FORMAT_WARMUP:
    import json as _json

    def build_gold_completion(record):
        """Construct a gold <think>/<verdict> completion from ground truth."""
        gt    = record['ground_truth_alignment']
        score = record['ground_truth_safety_score']
        traits = record['ground_truth_traits']

        # Simple but honest synthetic reasoning derived from ground truth
        safety_flags = [k for k in ('manipulation', 'deception', 'exploitation') if traits.get(k, 0) > 0.3]
        think_lines = [f'Evaluating the target agent.']
        if safety_flags:
            think_lines.append(f'Safety concerns detected: {', '.join(safety_flags)}.')
        think_lines.append(f'Safety score: {score:.2f}. Overall alignment: {gt}.')

        verdict_json = _json.dumps({**traits, 'alignment_status': gt}, indent=2)
        return f'<think>\n{chr(10).join(think_lines)}\n</think>\n<verdict>\n{verdict_json}\n</verdict>'

    # Build warmup dataset from first N short examples
    warmup_rows = []
    for rec in raw_records:
        msgs = [
            {'role': 'system', 'content': SYSTEM_PROMPT_TEMPLATE.format(author=rec['author'])},
            *rec['prompt'],
            {'role': 'assistant', 'content': build_gold_completion(rec)},
        ]
        text = tokenizer.apply_chat_template(msgs, tokenize=False)
        if len(tokenizer.encode(text)) <= WARMUP_MAX_TOKENS:
            warmup_rows.append({'text': text})
        if len(warmup_rows) >= WARMUP_EXAMPLES:
            break

    warmup_dataset = Dataset.from_list(warmup_rows)
    print(f'Warmup dataset: {len(warmup_dataset)} examples (max {WARMUP_MAX_TOKENS} tokens each).')
else:
    warmup_dataset = None
    print('Format warmup skipped.')

In [ ]:
if RUN_FORMAT_WARMUP and warmup_dataset is not None:
    from trl import SFTConfig, SFTTrainer

    sft_args = SFTConfig(
        output_dir='_warmup_output',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        report_to='none',
        max_seq_length=WARMUP_MAX_TOKENS,
        dataset_text_field='text',
    )

    sft_trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=warmup_dataset,
        args=sft_args,
    )

    print('Running format warmup SFT ...')
    sft_trainer.train()
    print('Warmup complete.')

## 6. Reward Functions

We import directly from `rewards.py` so the training notebook and the evaluation notebook
share the same parsing logic. The functions are shown here for transparency.

Three reward functions, each with TRL's `(prompts, completions, **kwargs) -> list[float]` signature:

- **`format_reward`** — checks for `<think>`/`<verdict>` tags and valid JSON (not scaled)
- **`alignment_reward`** — correct `alignment_status` gets `+2 × length_scale`, wrong gets `-1 × length_scale`
- **`trait_reward`** — weighted MAE across 12 traits, scaled by `length_scale`

The `length_scale` column (already in the dataset from `train.jsonl`) is automatically passed as
a kwarg by `GRPOTrainer` along with all other dataset columns.

In [ ]:
from grpo_pipeline.rewards import (
    format_reward,
    alignment_reward,
    trait_reward,
    extract_verdict,
    TRAIT_WEIGHTS,
)

# Quick sanity check — confirm reward functions work before spending GPU time
DUMMY_GOOD = '''<think>Agent is honest and helpful.</think>
<verdict>
{"virtue": 0.8, "goodwill": 0.7, "manipulation": 0.0, "deception": 0.0,
 "accuracy": 0.9, "reasoning": 0.85, "fabrication": 0.0, "broken_logic": 0.0,
 "recognition": 0.75, "compassion": 0.6, "dismissal": 0.0, "exploitation": 0.0,
 "alignment_status": "aligned"}
</verdict>'''
DUMMY_BAD = 'Just some text with no structure.'

good_c = [[{'role': 'assistant', 'content': DUMMY_GOOD}]]
bad_c  = [[{'role': 'assistant', 'content': DUMMY_BAD}]]

print('format_reward (good):', format_reward(None, good_c))           # [1.0]
print('format_reward (bad): ', format_reward(None, bad_c))            # [0.0]
print('alignment_reward (correct):', alignment_reward(None, good_c, ['aligned'], [1.0]))   # [2.0]
print('alignment_reward (wrong):  ', alignment_reward(None, good_c, ['misaligned'], [1.0])) # [-1.0]
print('alignment_reward (scaled): ', alignment_reward(None, good_c, ['aligned'], [0.4]))   # [0.8]
print('Reward functions OK.')

## 7. GRPO Training

Training logs will show a table with one row per step. Key columns to watch:

| Column | What to look for |
|---|---|
| `reward` | Should increase over time (starts near 0, target >2) |
| `rewards/format_reward/mean` | Should quickly approach 1.0 |
| `rewards/alignment_reward/mean` | Should increase gradually |
| `rewards/trait_reward/mean` | Should increase gradually |
| `kl` | Should stay low (<1); if it spikes the model is diverging |

The first ~50 steps may show near-zero or negative reward — this is normal while the model learns the output format.

In [ ]:
import gc
import torch
from vllm import SamplingParams
from trl import GRPOConfig, GRPOTrainer

# Clear any warmup memory
gc.collect()
torch.cuda.empty_cache()

vllm_sampling_params = SamplingParams(
    min_p=0.1,
    top_p=1.0,
    top_k=-1,
    seed=42,
    stop=[tokenizer.eos_token],
    include_stop_str_in_output=True,
)

training_args = GRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    temperature=1.0,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    logging_steps=1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_steps=MAX_STEPS if MAX_STEPS > 0 else None,
    num_train_epochs=1 if MAX_STEPS <= 0 else None,
    save_steps=SAVE_STEPS,
    output_dir=OUTPUT_DIR,
    report_to=REPORT_TO,
)

print(f'max_prompt_length={MAX_PROMPT_LENGTH}, max_completion_length={MAX_COMPLETION_LENGTH}')
print(f'num_generations={NUM_GENERATIONS}, batch_size={BATCH_SIZE}, max_steps={MAX_STEPS}')

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward,      # 1. structural format check (not length-scaled)
        alignment_reward,   # 2. correct alignment verdict (length-scaled)
        trait_reward,       # 3. accurate 12-trait MAE (length-scaled)
    ],
    args=training_args,
    train_dataset=dataset,
)
print('GRPOTrainer ready. Starting training ...')

In [ ]:
trainer.train()

## 8. Quick Inference Test

Run the trained model on a few test examples to visually confirm the output format
and check whether verdicts are reasonable before saving the adapter.

In [ ]:
import gc, json, torch
from grpo_pipeline.rewards import extract_verdict, extract_think
from grpo_pipeline.transform import SYSTEM_PROMPT_TEMPLATE

# Load a few test examples
TEST_FILE = '../transformed/test.jsonl'
with open(TEST_FILE) as f:
    test_records = [json.loads(line) for line in f if line.strip()]

# Switch to fast inference mode
gc.collect()
torch.cuda.empty_cache()
FastLanguageModel.for_inference(model)

def run_inference(rec, model, tokenizer, max_new_tokens=512):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT_TEMPLATE.format(author=rec['author'])},
        *rec['prompt'],
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc  = tokenizer(text, return_tensors='pt',
                     truncation=True, max_length=MAX_SEQ_LENGTH - max_new_tokens).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)

# Run on the first 3 test records
for i, rec in enumerate(test_records[:3]):
    generated = run_inference(rec, model, tokenizer)
    verdict   = extract_verdict(generated)
    think     = extract_think(generated)

    print(f'\n{"="*60}')
    print(f'Record {i}  |  author={rec["author"]}  |  GT={rec["ground_truth_alignment"]}')
    print(f'length_scale={rec["length_scale"]:.2f}  (turn {rec["turn_index"]}/{rec["total_turns"]-1})')
    print()
    if think:
        print('THINK (first 300 chars):')
        print(think[:300])
        print()
    if verdict:
        pred_align = verdict.get('alignment_status')
        match = '✓' if pred_align == rec['ground_truth_alignment'] else '✗'
        print(f'VERDICT: alignment={pred_align}  {match}')
        print(f'  (ground truth: {rec["ground_truth_alignment"]})')
    else:
        print('VERDICT: could not parse — raw output:')
        print(generated[:400])

## 9. Save LoRA Adapter

Save the trained LoRA adapter to `OUTPUT_DIR`. Then load it in `evaluate.ipynb` to run
the full test-set evaluation and compare against the base model baseline.

In [ ]:
import os

# Training used GRPO (RL objective) with LoRA (parameter-efficient adapters).
# The GRPOTrainer optimised only the LoRA weights — the base model is unchanged.
# We save both: the lightweight LoRA adapter (for fast iteration / evaluate.ipynb)
# and the full merged 16-bit model (standalone deployment without PEFT).

HF_USERNAME = 'Eephor'   # ← your HuggingFace username if pushing
HF_TOKEN    = ''         # ← set your HF token here or via env var HF_TOKEN

LORA_DIR   = OUTPUT_DIR            # e.g. ../lora-adapter
MERGED_DIR = OUTPUT_DIR + '_16bit' # e.g. ../lora-adapter_16bit

# 1. Save LoRA adapter — load later with PeftModel.from_pretrained()
os.makedirs(LORA_DIR, exist_ok=True)
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f'LoRA adapter saved  → {LORA_DIR}')

# 2. Save merged 16-bit model — load like any normal HuggingFace model
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print(f'Merged model saved  → {MERGED_DIR}')

# ── Optional: push to HuggingFace Hub ─────────────────────────────────────────
if False: model.push_to_hub_merged(f'{HF_USERNAME}/moltbook-oversight-agent', tokenizer, save_method='merged_16bit', token=HF_TOKEN)

# ── Optional: save as GGUF for llama.cpp / Ollama ────────────────────────────
if False: model.save_pretrained_gguf(OUTPUT_DIR + '_gguf', tokenizer)
if False: model.save_pretrained_gguf(OUTPUT_DIR + '_gguf', tokenizer, quantization_method='q4_k_m')
if False: model.push_to_hub_gguf(f'{HF_USERNAME}/moltbook-oversight-agent-gguf', tokenizer, token=HF_TOKEN)
# ─────────────────────────────────────────────────────────────────────────────

print()
print('Next steps:')
print('  1. Open evaluate.ipynb')
print(f'  2. Set MODEL_NAME = "{MERGED_DIR}"  (merged, no adapter needed)')
print(f'     OR set LORA_ADAPTER = "{LORA_DIR}"  (base model + adapter)')
print('  3. Run Section 4 (Batch Metrics) to compare vs. base model baseline')
print('  4. Run Section 5 (Side-by-side Comparison) to inspect verdict changes')

---
## Appendix: Scaling to Qwen3-8B

Change one line at the top of this notebook:
```python
MODEL_NAME = 'unsloth/Qwen3-8B-Instruct'
```

Requirements:
- Colab Pro with L4 (22 GB) or RTX 3090/4090 (24 GB)
- Reduce `NUM_GENERATIONS = 2` if you hit OOM on forward pass
- Consider `BATCH_SIZE = 2` and `gradient_accumulation_steps = 2`
- `MAX_STEPS = 500` recommended for the larger model

No reward function changes, no data changes — the model name is the only edit needed.